## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# 08 - Evaluation, control experiments, comparison with all earlier runs, export
Discrimination + calibration, **paired control experiments** (is it the CALF numbers, the matched precedent, or the smoother layout?),
counterfactual consistency, audit coverage-risk, refutations, and a table across every earlier run stored in the dataset's `results/`.
The last cell exports this run so it can be added to `results/` for future comparisons.

In [ ]:
import json, pickle, numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from causal_rag import eval as ev, compare, refute
from causal_rag.config import Featurizer, fit_medians, add_features, TREATMENTS
sc = pd.read_parquet(paths.art('llm_scores.parquet')); cfdf = pd.read_parquet(paths.art('cf_audit.parquet'))
recs = [json.loads(l) for l in open(paths.art('decisions.jsonl'))]
y = sc.default.values
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr=(df.role=='train').values; va=(df.role=='val').values
d = add_features(df, fit_medians(df, tr)); yy = d.default.values
flat = Featurizer(None).fit(d, tr); X = np.hstack(flat.transform(d))
te_idx = d.set_index('case_id').index.get_indexer(sc.case_id.values)
gbm = ev.gbm_baseline(X[tr], yy[tr], X[va], yy[va], X[te_idx])
calf = pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id').loc[sc.case_id.values]
calf_risk = np.mean([calf[f'pf_{tk}'].values for tk in TREATMENTS], axis=0)
LLM_SYS = [c for c in sc.columns if c not in ('case_id','default')]
scores = {k: sc[k].values for k in LLM_SYS}
scores.update({'CALF factual risk (tabular)': calf_risk, 'LightGBM (tabular ceiling)': gbm})
B = 300 if SMOKE else 1000
disc = ev.discrimination_table(y, scores, ref='no_rag', B=B); display(disc)
if len(y) > 300:    # the first 300 applicants are the SAME ones scored in v1.0 / v1.1 (same seed) -> like-for-like view
    disc300 = ev.discrimination_table(y[:300], {k: v[:300] for k, v in scores.items()}, ref='no_rag', B=B)
    disc300.to_csv(paths.art('discrimination_first300.csv'), index=False); print('first 300 applicants only (comparable with earlier runs):'); display(disc300)
pairs = [('causal_rag_smooth','no_rag'), ('causal_rag','no_rag'), ('causal_rag_smooth','causal_rag'),
         ('causal_rag','causal_calf_only'), ('causal_rag','causal_precedent_only'),
         ('causal_calf_only','no_rag'), ('causal_precedent_only','no_rag'), ('causal_rag_smooth','knn_rag'), ('causal_rag_smooth','vanilla_rag')]
pw = ev.pairwise_table(y, scores, pairs, B=B); pw.to_csv(paths.art('pairwise_table.csv'), index=False); display(pw)
thr_by = {s: float(np.quantile(sc[s].values, 0.8)) for s in LLM_SYS}
cft = ev.counterfactual_table(cfdf, thr_by); cft.to_csv(paths.art('counterfactual_table.csv'), index=False); display(cft.T)
cov = ev.coverage_risk(y, [r['decision'] for r in recs]); json.dump(cov, open(paths.art('coverage_risk.json'),'w'), indent=1); print(cov)
ref = pd.read_csv(paths.art('refutations.csv')) if paths.have('refutations.csv') else None
if ref is not None: display(refute.pass_rates(ref))
disc.to_csv(paths.art('results_table.csv'), index=False)
json.dump(dict(run=RUN_NAME, audit_system=AUDIT_SYSTEM, discrimination=disc.to_dict('records'), counterfactual=cft.to_dict('records'),
               audit=cov, n_eval=len(y), default_rate=float(y.mean())), open(paths.art('metrics.json'), 'w'), indent=1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
disc.set_index('system').auroc.plot.barh(ax=ax[0]); ax[0].set_title('AUROC (2015 hold-out sample)'); ax[0].axvline(.5, c='k', ls=':')
cft.set_index('system')[['sign_agree_all','sign_agree_responders','trivial_prior_agree']].plot.bar(ax=ax[1]); ax[1].set_title('flip direction agrees with CALF sign'); ax[1].axhline(.5, c='k', ls=':')
plt.tight_layout(); plt.savefig(paths.art('summary.png'), dpi=130); plt.show()
# ---- history across runs (uses the dataset's results/ folder) + export of this run
runs = compare.all_runs(RUN_NAME, disc, cft, cov)
print('earlier runs found:', list(compare.previous_runs()))
display(runs['discrimination'].pivot_table(index='system', columns='run', values='auroc').round(3))
display(runs['coverage'].set_index('run')[['n','approve_rate','decline_rate','refer_rate','default_rate_among_approved','default_rate_among_referred']].round(3))
out = compare.export_run(RUN_NAME, disc, cft, cov, dict(N_EVAL=N_EVAL, CHUNK=CHUNK, EPOCHS=EPOCHS, AUDIT_SYSTEM=AUDIT_SYSTEM, WITH_ECONML=WITH_ECONML, code_version='1.2.0'),
                         extra_files=[paths.art(n) for n in ('pairwise_table.csv','discrimination_first300.csv','calf_results.csv','rct_results.csv','refutations.csv','summary.png','metrics.json')])
print('exported to', out, '-> copy this folder into the dataset under results/ to keep the run')

### Demo (optional, needs GPU; prints a public Gradio link)

In [ ]:
import pickle, json
from causal_rag import demo, retrieval as rt, embedding, causal_graph as cgm, pipeline as pl
from causal_rag.llm import LLM
g, adj = cgm.load_graph()
medians = fit_medians(df, tr); idx = d[d.role.isin(['train','val'])].reset_index(drop=True)
emb = embedding.Embedder()
R = rt.RetrieverAgent(idx, np.load(paths.art('case_emb.npy')).astype('float32'), pd.read_parquet(paths.art('policy_chunks.parquet')),
                      np.load(paths.art('policy_emb.npy')).astype('float32'),
                      pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id'), emb, adj)
m = sc[AUDIT_SYSTEM].values; thr = float(np.quantile(m, .8)); band = float(.15*m.std())
demo.launch(LLM(), R, pl.CalfScorer(), g, d[d.role=='test'].reset_index(drop=True), medians, thr, band, system=AUDIT_SYSTEM)